# Step 3 — Milvus Lite: Schema, Index & Ingest
Builds a Milvus Lite collection with:
- `dense_vec`  — HNSW index (cosine)
- `sparse_vec` — SPARSE_INVERTED_INDEX (BM25 weights)
- Scalar fields for metadata filtering: `year`, `naics3`, `section`, `ticker`

In [ ]:
!pip install -q milvus-lite 'pymilvus>=2.4.0'

from google.colab import drive
drive.mount('/content/drive')

import os, json, pickle
import numpy as np

BASE_DIR    = '/content/drive/MyDrive/mdna_rag'
CHUNKS_IN   = f'{BASE_DIR}/chunks.jsonl'
DENSE_IN    = f'{BASE_DIR}/embeddings_dense.npz'
SPARSE_IN   = f'{BASE_DIR}/embeddings_sparse.pkl'
MILVUS_DB   = f'{BASE_DIR}/milvus_mdna.db'
COLLECTION  = 'mdna_2024_25'
DENSE_DIM   = 768
print('✅ Paths set')

In [ ]:
from pymilvus import (
    MilvusClient, DataType,
    CollectionSchema, FieldSchema,
)

client = MilvusClient(MILVUS_DB)

# Drop if re-running
if client.has_collection(COLLECTION):
    client.drop_collection(COLLECTION)
    print(f'Dropped existing collection: {COLLECTION}')

# ── Schema ────────────────────────────────────────────────────────────────────
schema = MilvusClient.create_schema(auto_id=True, enable_dynamic_field=False)

schema.add_field('id',          DataType.INT64,        is_primary=True)

# Metadata — all filterable
schema.add_field('ticker',      DataType.VARCHAR,      max_length=16)
schema.add_field('cik',         DataType.VARCHAR,      max_length=32)
schema.add_field('accession',   DataType.VARCHAR,      max_length=64)
schema.add_field('year',        DataType.INT16)
schema.add_field('sic',         DataType.INT32)
schema.add_field('naics3',      DataType.INT16)
schema.add_field('naics_label', DataType.VARCHAR,      max_length=128)
schema.add_field('section',     DataType.VARCHAR,      max_length=64)
schema.add_field('chunk_index', DataType.INT16)
schema.add_field('token_count', DataType.INT16)
schema.add_field('text',        DataType.VARCHAR,      max_length=4096)

# Vectors
schema.add_field('dense_vec',   DataType.FLOAT_VECTOR, dim=DENSE_DIM)
schema.add_field('sparse_vec',  DataType.SPARSE_FLOAT_VECTOR)

# ── Index params ──────────────────────────────────────────────────────────────
index_params = client.prepare_index_params()

index_params.add_index(
    field_name  = 'dense_vec',
    index_type  = 'HNSW',
    metric_type = 'COSINE',
    params      = {'M': 16, 'efConstruction': 200},
)
index_params.add_index(
    field_name  = 'sparse_vec',
    index_type  = 'SPARSE_INVERTED_INDEX',
    metric_type = 'IP',         # inner product for sparse BM25 weights
    params      = {'drop_ratio_build': 0.2},  # prune low-weight tokens
)

client.create_collection(
    collection_name = COLLECTION,
    schema          = schema,
    index_params    = index_params,
)
print(f'✅ Collection created: {COLLECTION}')

In [ ]:
# ── Load chunks + embeddings ──────────────────────────────────────────────────
chunks = []
with open(CHUNKS_IN) as f:
    for line in f:
        chunks.append(json.loads(line))

dense_matrix = np.load(DENSE_IN)['embeddings']   # (N, 768) float32

with open(SPARSE_IN, 'rb') as f:
    sp_data = pickle.load(f)
sparse_vecs = sp_data['sparse_vecs']              # list of {int: float} dicts

assert len(chunks) == len(dense_matrix) == len(sparse_vecs), 'Count mismatch!'
print(f'✅ {len(chunks):,} records loaded')

In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 1000
inserted   = 0

def build_batch(start: int, end: int) -> list[dict]:
    rows = []
    for i in range(start, end):
        c = chunks[i]
        rows.append({
            'ticker'      : c['ticker'][:16],
            'cik'         : str(c['cik'])[:32],
            'accession'   : str(c['accession'])[:64],
            'year'        : int(c['year']),
            'sic'         : int(c['sic']),
            'naics3'      : int(c['naics3']),
            'naics_label' : c['naics_label'][:128],
            'section'     : c['section'][:64],
            'chunk_index' : int(c['chunk_index']),
            'token_count' : int(c['token_count']),
            'text'        : c['text'][:4096],
            'dense_vec'   : dense_matrix[i].tolist(),
            'sparse_vec'  : sparse_vecs[i],   # {token_id: weight}
        })
    return rows

for start in tqdm(range(0, len(chunks), BATCH_SIZE), desc='Ingesting to Milvus'):
    end   = min(start + BATCH_SIZE, len(chunks))
    batch = build_batch(start, end)
    res   = client.insert(collection_name=COLLECTION, data=batch)
    inserted += res['insert_count']

print(f'\n✅ Inserted {inserted:,} records into {COLLECTION}')

In [ ]:
# ── Load collection into memory & verify ─────────────────────────────────────
client.load_collection(COLLECTION)
stats = client.get_collection_stats(COLLECTION)
print(f'Collection row count: {stats["row_count"]:,}')

# Quick metadata check
sample = client.query(
    collection_name = COLLECTION,
    filter          = 'year == 2024',
    output_fields   = ['ticker','year','naics3','naics_label','section'],
    limit           = 3,
)
for r in sample:
    print(r)